# Chapter 00-02 · What machine learning is, what it is not, and when a rule wins

**Label:** Core  |  **Time:** ~45 minutes  |  **Difficulty:** gentle

**Prerequisites:** 00-01. You should be able to say what a prediction rule, a feature and a
target are, and why a baseline is compulsory.

**Position in the learning path:** module 00 (Orientation), chapter 2 of 3. Before this:
**00-01**. After this: **00-03**, on the kinds of learning and on prediction versus
explanation versus cause.

---

## Why this matters

The most valuable sentence in this course is **"this doesn't need machine learning."**

It is valuable because it is rare. Saying it saves months of work, a pile of infrastructure
and a permanent maintenance burden. It also happens to be a question you will be asked
directly in interviews - *"when would you not use ML?"* - and answered badly by most
candidates, who reach for a slogan about small data.

The real answer is sharper than that, and it comes from understanding exactly what trade you
make when you choose machine learning. This chapter makes that trade explicit, by building
the same task twice: once where the trade is a clear loss, and once where it is a clear win.

## What you will be able to do

By the end of this chapter you can:

1. **State** what machine learning is, in terms of what it inverts about ordinary programming.
2. **Apply** a seven-question checklist to decide whether a task needs machine learning.
3. **Name** five alternatives to ML and the situation each one beats it in.
4. **Demonstrate** a task where a three-line rule beats a trained model, and say precisely why.
5. **Diagnose** what happens to a rule and to a model when the world changes underneath them,
   and compare the cost of repairing each.

## Warm-up: retrieve, do not reread

Answer from memory, out loud, before scrolling. Trying to recall something is what fixes it;
rereading only feels like it does.

1. What is a prediction rule?
2. What is the one question you must ask of every candidate feature?
3. Why is a training-set score not evidence?
4. What are the units of MAE if the target is bikes?
5. Rule C in the last chapter scored MAE 0.00 on the days it was built from. What was wrong
   with it?

<br>

*Answers: (1) anything that turns a row's features into a guess about its target. (2) will I
actually have this value at the moment I make the prediction? (3) it measures memory of rows
already seen, not skill on new ones. (4) bikes. (5) it had memorised the answers by day
number, so on any new day it fell back to the overall average - a perfect score and no skill.*

## The situation

Tomás runs the café in the same park as Maria's bike stand. He has read that he should be
"using AI", and he arrives with four things he wants a model for:

1. *"Tell me which customers get the loyalty discount."* The policy is written on a card
   behind the till: spend over 100 EUR in a month and you get 10% off.
2. *"Tell me how much coffee I sold last July."* It is all in the till system.
3. *"Tell me whether putting the pastries by the door increases sales."* He has never tried it.
4. *"Tell me how many croissants to bake tomorrow morning."* Nobody knows; some mornings he
   throws thirty away and some mornings he sells out by nine.

Exactly one of these is a machine learning problem.

**The question this chapter answers:** how do you tell which one - fast, and for any task
someone brings you?

In [ ]:
import pandas as pd

candidates = pd.DataFrame({
    "task": [
        "who gets the loyalty discount",
        "how much coffee sold last July",
        "do pastries by the door lift sales",
        "how many croissants to bake tomorrow",
        "is this email address valid",
        "which of these two menu designs sells more",
    ],
    "is_the_rule_known": ["yes", "n/a", "no", "no", "yes", "no"],
    "about_the_past_or_the_future": ["either", "past", "future", "future", "either", "future"],
    "question_type": [
        "apply a policy", "look something up", "does X cause Y",
        "predict a number", "check a format", "does X cause Y",
    ],
})
candidates

### Predict before running

Before reading on, decide for yourself, and write down your reason in five words:

1. Which of the six tasks needs machine learning?
2. For each of the others, what should be used instead?
3. Tomás says *"but I have three years of till data, surely that means ML applies."* Is having
   data a reason to use machine learning?

## What machine learning actually is

Ordinary programming and machine learning use the same three ingredients in different places.

```
   ordinary programming            machine learning
   -------------------            ----------------
   rules   +  data                data  +  answers
        |                              |
        v                              v
     answers                         rules
```

In ordinary programming **you** write the rule, and the computer applies it to data to produce
answers. In machine learning you supply data **together with the answers**, and the computer
searches for a rule that would have produced those answers.

That is the whole inversion. Everything else - trees, gradients, networks - is detail about
*how* the search is done.

Read the diagram once more, because two consequences fall straight out of it and they are the
consequences that matter:

- **You need the answers.** Not just data: data *labelled with what actually happened*. Tomás
  has three years of till data, which tells him what he sold, not how many croissants he
  *should* have baked. Those are different columns, and only one of them exists.
- **You get back an approximation of a rule, not the rule.** The search finds something that
  fits the examples it was shown. If you already possess the exact rule, running this search
  can only lose accuracy, never gain it.

That second point sounds abstract. It is not, and the next section makes it concrete enough to
argue with.

## Task A: a rule you already have

The café's loyalty policy, written on the card behind the till:

> **Free delivery if the order total is at least 50 EUR *and* the address is domestic.**

Apply it by hand to five orders. This takes twenty seconds and is the point of the exercise.

| Order | Total (EUR) | Destination | Free delivery? |
|---|---|---|---|
| 1 | 72.00 | domestic | ? |
| 2 | 49.99 | domestic | ? |
| 3 | 120.00 | international | ? |
| 4 | 50.00 | domestic | ? |
| 5 | 33.50 | international | ? |

*Answers: yes, no (one cent short), no (wrong country, however large), yes (exactly 50 counts,
because the policy says "at least"), no.*

Order 4 is the interesting one. The policy has a **sharp edge** at exactly 50.00, and it is
sharp because someone decided it should be. Keep that edge in mind.

Now: 300 orders, labelled by that policy. We train two models on 200 of them and check all
three approaches on the remaining 100.

In [ ]:
import numpy as np

rng = np.random.default_rng(4)

n_orders = 300
order_total = np.round(rng.uniform(5, 120, n_orders), 2)     # EUR
is_domestic = rng.random(n_orders) < 0.70

# The policy. Three lines, exact, no data required.
free_delivery = (order_total >= 50) & is_domestic            # SYNTHETIC labels

orders = pd.DataFrame({"order_total": order_total,
                       "is_domestic": is_domestic.astype(int),
                       "free_delivery": free_delivery})
print("share of orders with free delivery:", round(free_delivery.mean(), 3))
orders.head()

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier

features = ["order_total", "is_domestic"]
train, test = orders.iloc[:200], orders.iloc[200:]

logreg = LogisticRegression().fit(train[features], train["free_delivery"])
tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(train[features], train["free_delivery"])

# The policy needs no training at all.
policy = (test["order_total"] >= 50) & (test["is_domestic"] == 1)

for name, pred in [("the written policy (3 lines)", policy),
                   ("decision tree (200 examples)", tree.predict(test[features])),
                   ("logistic regression (200 examples)", logreg.predict(test[features]))]:
    print(f"{name:<36} accuracy = {accuracy_score(test['free_delivery'], pred):.3f}")

**Accuracy** is simply the share of predictions that are correct - 0.97 means 97 of every 100
orders were classified right. It is dimensionless, and it is a crude measure that hides a great
deal; chapter 06-04 shows exactly how it lies. It is adequate here only because the two
outcomes are reasonably balanced (about 47% of orders qualify) and because both kinds of
mistake cost about the same.

Now read the three numbers.

**The written policy: 1.000.** Of course. It *is* the definition of the answer. It cannot be
wrong.

**The decision tree: 1.000.** It found the rule. This is machine learning doing as well as it
can possibly do on this task - and the very best it can do is *reproduce something you already
had*, in exchange for 200 labelled examples, a training step, a library dependency, and an
object somebody now has to store, version and monitor.

**Logistic regression: 0.970.** Three orders in a hundred, wrong. Not because logistic
regression is bad, but because it draws a single straight boundary and this policy is not a
straight boundary - it is a threshold that applies only to domestic orders. The model can
approximate that shape; it cannot represent it.

That last line is the general lesson, and it will come back in every modelling chapter: **a
model can only express certain shapes, and if the truth is not one of them the model
approximates it.** Approximation is a wonderful bargain when you do not know the truth. It is
a pure loss when you do.

In [ ]:
import matplotlib.pyplot as plt

wrong = logreg.predict(test[features]) != test["free_delivery"]
jitter = rng.normal(0, 0.04, len(test))

fig, ax = plt.subplots(figsize=(7, 3.6))
for label, colour, marker in [(1, "#0072B2", "o"), (0, "#D55E00", "x")]:
    m = test["free_delivery"] == label
    ax.scatter(test.loc[m, "order_total"], test.loc[m, "is_domestic"] + jitter[m.to_numpy()],
               color=colour, marker=marker, s=45,
               label="free delivery" if label else "no free delivery")
ax.scatter(test.loc[wrong.to_numpy(), "order_total"],
           test.loc[wrong.to_numpy(), "is_domestic"] + jitter[wrong.to_numpy()],
           facecolors="none", edgecolors="black", s=190, linewidths=1.6,
           label="logistic regression got it wrong")
ax.axvline(50, color="grey", linestyle="--", label="the policy: 50 EUR")
ax.set_yticks([0, 1], ["international", "domestic"])
ax.set_xlabel("Order total (EUR)")
ax.set_title("100 held-out orders: where the model disagrees with the policy")
ax.legend(loc="center right", fontsize=8)
plt.show()

Look at where the black rings are.

Every mistake sits close to the boundary the policy defines, and on the side of it where a
straight line has to compromise between the two destination groups. The model has not
misunderstood the business in some deep way; it has smoothed a sharp edge, because a smooth
thing is what it is made of.

If you ever have to explain to a customer why their 51 EUR domestic order was charged
delivery, "the model estimated a low probability" is not an answer anyone will accept, and in
some domains it is not an answer the law will accept either. The policy, by contrast, explains
itself.

---

## Task B: a rule nobody has

Same shape of problem - a yes/no prediction - but now the question is *"will this parcel
arrive late?"*

Nobody wrote a policy for this. Lateness comes from distance, weight, whether it is a holiday
week, the weather, and a great deal that is not recorded anywhere. There is no card behind the
till to read.

The data below is **SYNTHETIC**: 800 deliveries whose lateness we generate from a known
combination of four causes plus randomness. We know the true recipe, which is exactly why it
is useful - we can see whether the methods recover it.

In [ ]:
rng_b = np.random.default_rng(7)

n_parcels = 800
distance_km = rng_b.uniform(1, 900, n_parcels)
weight_kg = rng_b.uniform(0.1, 20, n_parcels)
holiday_week = rng_b.random(n_parcels) < 0.20
bad_weather = rng_b.random(n_parcels) < 0.30

# Several weak causes, plus noise that nothing could predict.  SYNTHETIC.
pressure = (-3.9 + 0.004 * distance_km + 0.10 * weight_kg
            + 1.3 * holiday_week + 1.0 * bad_weather + rng_b.normal(0, 0.8, n_parcels))

parcels = pd.DataFrame({
    "distance_km": distance_km.round(1),
    "weight_kg": weight_kg.round(2),
    "holiday_week": holiday_week.astype(int),
    "bad_weather": bad_weather.astype(int),
    "late": pressure > 0,
})
print("share of parcels that arrived late:", round(parcels['late'].mean(), 3))
parcels.head()

In [ ]:
parcel_features = ["distance_km", "weight_kg", "holiday_week", "bad_weather"]
p_train, p_test = parcels.iloc[:600], parcels.iloc[600:]

# Baseline, as always: the answer that needs no information at all.
always_on_time = np.zeros(len(p_test), dtype=bool)

# The best single hand-written rule we can find: one threshold on distance,
# with the threshold chosen using training data only.
thresholds = np.arange(50, 900, 10)
scores = [accuracy_score(p_train["late"], p_train["distance_km"] > t) for t in thresholds]
best_threshold = thresholds[int(np.argmax(scores))]

model = LogisticRegression(max_iter=1000).fit(p_train[parcel_features], p_train["late"])

for name, pred in [("baseline: always 'on time'", always_on_time),
                   (f"hand rule: distance > {best_threshold} km", p_test["distance_km"] > best_threshold),
                   ("logistic regression (4 features)", model.predict(p_test[parcel_features]))]:
    print(f"{name:<36} accuracy = {accuracy_score(p_test['late'], pred):.3f}")

Now the numbers point the other way.

The baseline gets **0.705** - because most parcels are on time, so refusing to think is already
right seven times in ten. The best single hand-written threshold manages **0.730**, barely
better. The model gets **0.855**.

Why the gap? Because lateness is not caused by one thing crossing one line. It is four weak
influences adding up, and a human writing rules would need to nest conditions - *over 600 km,
or over 400 km in a holiday week, or over 300 km in bad weather with a heavy parcel...* -
guessing at each threshold. That is precisely the search machine learning does, and it does it
better than a person guessing, because it can weigh all four at once against 600 examples.

**This is the trade, stated plainly:**

> Machine learning is worth it when the rule is **unknown to you but present in the data**,
> and when an approximate answer is genuinely useful.

Task A failed both halves: the rule was known, and an approximation of a known rule is a
downgrade. Task B passes both.

*One honest caveat about the numbers above.* Accuracy is a defensible lens here only because
the classes are not wildly imbalanced (36% late) and because we are pretending a late parcel
and a false alarm cost the same. They almost never do. Chapter 06-05 replaces accuracy with
measures that respect the difference, and 06-07 chooses the threshold from the actual cost of
each kind of mistake.

## The seven questions

Ask these in order. A "no" to any of 1-4 usually means machine learning is the wrong tool -
not a harder ML problem, a different tool.

1. **Is there a pattern at all?** Does the answer really depend on information you have?
   Nothing learns from noise. If a domain expert cannot say *anything* about which cases differ,
   be suspicious before you begin.
2. **Can you write the rule down?** If yes, write it down. You get exactness, an explanation,
   an audit trail, and no training data. This is the question people skip.
3. **Do you have examples with the answers attached?** Not "data" - *outcomes*. Three years of
   sales tells Tomás what he sold, not what he should have baked. Fixing that usually means
   changing what you record, which is a different project with a different budget.
4. **Will the future resemble the past?** A learned rule is a summary of what already happened.
   If the process changes - new competitor, new pricing, new law, pandemic - the summary
   describes a world that has gone.
5. **Is being wrong sometimes acceptable?** Every model is wrong on some cases. If a single
   wrong answer is catastrophic, or must be justified to a court or a regulator, the bar is much
   higher and often not met.
6. **Is the decision repeated often enough to be worth it?** A decision made four times a year
   should be made by a person with a spreadsheet. Automation pays for itself through volume.
7. **Will the inputs exist at decision time?** The question from 00-01, and still the one that
   kills the most projects. A feature that only exists in the historical table is not a feature.

Applied to Tomás:

| His request | Verdict | Because |
|---|---|---|
| who gets the loyalty discount | **not ML** - write the rule | question 2: the rule is on a card |
| coffee sold last July | **not ML** - a database query | it is about the past; it is *recorded*, not predicted |
| do pastries by the door lift sales | **not ML** - run an experiment | it asks about a *cause*, not a prediction |
| how many croissants tomorrow | **ML is reasonable** | rule unknown, outcomes recorded daily, wrong-by-a-few is fine, decision is daily |
| is this email valid | **not ML** - a format check | question 2 again, and a spec exists |
| which menu design sells more | **not ML** - an A/B test | a cause again, and a comparison you can measure directly |

## What to use instead

Five alternatives, and the situation each one wins in. Recognising these is most of the skill.

| Instead of ML | Use it when | What you gain |
|---|---|---|
| **A written rule / `if` statement** | The policy exists, or an expert can state it | Exactness, explainability, one-line changes, no data needed |
| **A database query or a dashboard** | The question is about what *did* happen | The true answer instead of an estimate of it |
| **A designed experiment (A/B test)** | The question is "does X *cause* Y" | An answer to the question actually asked - see 00-03 and 12-07 |
| **Optimisation / operations research** | You know the objective and constraints and want the *best plan* | A provably good decision, not a guess about an outcome |
| **A mechanistic model (physics, accounting)** | The mechanism is known and stable | Extrapolation beyond your data, which learned models cannot do |

Two more that deserve saying out loud because they are the most commonly skipped:

- **Better measurement.** Very often a prediction problem exists only because nobody records
  the thing. Instrumenting it beats predicting it, every time.
- **A person.** For low volume, high stakes, or genuinely novel cases, a human with good
  information is faster to set up, easier to correct, and accountable in a way software is not.

And the honest converse, so this chapter is not simply anti-ML: when the rule really is unknown,
the volume is high and the mistakes are survivable - fraud scoring, demand forecasting, text
sorting, recommendation, image tagging - hand-written rules do not merely lose. They become
unmaintainable thickets that nobody dares to edit. Both failure modes are real.

---

## Failure lab: the day the policy changed

The café changes its offer. From the first of the month, free delivery starts at **40 EUR**
instead of 50. The card behind the till is rewritten.

Nobody tells the model.

**Predict before running:** the model reproduced the old policy perfectly on held-out data.
What accuracy will it get on orders that follow the new policy - roughly? And what would it
take to fix each of the three approaches?

In [ ]:
rng_new = np.random.default_rng(99)

m = 150
new_total = np.round(rng_new.uniform(5, 120, m), 2)
new_domestic = rng_new.random(m) < 0.70
new_truth = (new_total >= 40) & new_domestic          # the NEW policy
new_X = pd.DataFrame({"order_total": new_total, "is_domestic": new_domestic.astype(int)})

for name, pred in [("model trained before the change", tree.predict(new_X)),
                   ("the old rule, left alone", (new_total >= 50) & new_domestic),
                   ("the rule, one number edited", (new_total >= 40) & new_domestic)]:
    print(f"{name:<34} accuracy = {accuracy_score(new_truth, pred):.3f}")

### Diagnosis

Both the stale model and the stale rule score **0.947**. Every mistake is a domestic order
between 40 and 50 EUR - a customer who was promised free delivery and charged for it.

Notice first what this does *not* show. It does not show that rules are immune to change; the
old rule broke in exactly the same way and by exactly the same amount. Anything that encodes a
policy goes wrong when the policy moves. That symmetry is the honest half of the story, and
skipping it would be a cheap argument.

The difference is what happens next.

| | Stale written rule | Stale learned model |
|---|---|---|
| **How you notice** | You changed the policy, so you know the rule mentions it | Nothing announces it. You notice when a customer complains, or never |
| **How you diagnose** | Read three lines | Inspect a fitted object to work out what boundary it learned |
| **How you fix** | Edit one number | Collect labelled data *under the new policy*, retrain, validate, redeploy |
| **How long until correct** | Minutes | As long as it takes to accumulate new labelled examples |
| **What could go wrong in the fix** | A typo | Retraining on a mix of old and new policy data, learning a blurred boundary |

The last row is the one that bites in practice. Retrain too early and your training data is
mostly the *old* world; the model learns a compromise between two policies and is now wrong in
a way that is much harder to spot than being wrong in one clean place.

This general problem - the world moving away from the data a model was fitted to - is called
**drift**, and it is the reason that deploying a model is the beginning of the work rather
than the end. Chapter 13-08 is about detecting it and 12-02 about responding to it.

### Remedies, and what each costs

| Remedy | What it fixes | What it costs | When to prefer it |
|---|---|---|---|
| Do not model a known policy at all | The whole failure | Nothing. You save work | Whenever question 2 answers "yes" |
| Encode the policy, model only the residual | Keeps the sharp edge exact | More design effort | Policy plus messy extras, e.g. delivery time given a fixed fee rule |
| Monitor accuracy on fresh labelled data | Detection, not prevention | A labelling process that never stops | Any deployed model. Not optional |
| Retrain on a schedule | Slow drift | Compute, and a validation step each time | Gradual change, e.g. customer habits |
| Retrain on a trigger | Sudden drift | Needs a monitor that works first | Known events: policy, pricing, product changes |

## Common misconceptions

**"We have a lot of data, so we should use machine learning."**
Data volume is a *prerequisite* for ML, not a reason for it. Tomás's three years of till
records make his loyalty policy no less exactly known. Ask question 2 before question 3.

**"Machine learning is when a computer decides something automatically."**
An `if` statement decides automatically too. The distinguishing feature is not automation but
where the rule came from: written by a person, or searched for in labelled examples.

**"Rules are old-fashioned; models are modern."**
Every serious production system is mostly rules, with models where rules genuinely fail.
Payment systems, tax software and safety interlocks are rule-based on purpose, because
exactness and auditability are worth more there than a few points of accuracy.

**"If the model beats the rule on accuracy, use the model."**
Only if accuracy is the whole cost. The rule may be free to run, instantly changeable,
explainable to a customer, and legally defensible. Those are real values that no accuracy
number contains.

**"ML can find the answer even if we never recorded it."**
It cannot. If the outcome column does not exist, there is nothing to learn from. This is the
most expensive misunderstanding on the list, because it is usually discovered three weeks into
a project.

**"We'll use ML to find out what causes this."**
A model finds what *predicts*. Umbrella sales predict rain beautifully and cause none of it.
This deserves a chapter of its own, and it gets one: **00-04**.

---

## Exercises

Solutions with reasoning: `solutions/00_orientation/00-02_what_ml_is_solutions.ipynb`.

### Quick understanding

**E1 (define).** In two sentences, state what machine learning inverts about ordinary
programming, and name the one extra ingredient it requires.

**E2 (explain).** Task A's decision tree scored a perfect 1.000. Explain, in three sentences,
why that is *not* a reason to deploy it.

**E3 (define).** What is accuracy, and why is it a poor description of a model that predicts
whether a rare disease is present?

### Hand calculation

**E4 (calculate).** The new policy is "free delivery at 40 EUR or more, domestic only". Apply
it by hand to: (a) 39.99 domestic, (b) 40.00 domestic, (c) 40.00 international, (d) 250.00
international, (e) 45.00 domestic. Then say which of these five the *old* model would get
wrong, and why.

**E5 (calculate).** In a test set of 200 parcels, a model predicts "late" for 60 parcels and is
right about 45 of them; 30 parcels that were actually late were predicted on time. Compute the
accuracy. Then compute the accuracy of "always predict on time" on the same 200 parcels. What
do you conclude, and what extra information would you need before recommending the model?

### Coding

**E6 (code).** In task A, retrain the decision tree using only **20** training orders instead
of 200. Score it against the policy on the same 100 held-out orders. Repeat for 50 and 100.
Plot accuracy against training size. What does the shape tell you about what data buys you?

**E7 (design + code).** In task B, find the best **two**-condition hand rule of the form
`distance > a OR (bad_weather AND distance > b)`, choosing `a` and `b` on the training set
only. How close does it get to the model? What does the gap suggest about when hand rules stop
being worth writing?

### Interpretation

**E8 (interpret).** A colleague reports: *"our churn model is 94% accurate, the rule-based
system was 91%."* Give three questions you would ask before agreeing that the model should
replace the rules.

### Debugging

**E9 (diagnose).** A model that predicts whether an invoice will be paid late has been in
production for eight months. Accuracy on the monthly reports has slipped from 0.88 to 0.79.
List four possible causes, ordered by how cheap they are to check, and say how you would check
the first two.

### Exam and interview reasoning

**E10 (defend).** *"When would you not use machine learning?"* Answer in under 90 seconds of
speech - roughly 150 words - using at least three of the seven questions and one concrete
example.

**E11 (design).** A bank asks you to build a model that decides whether to approve a loan. List
three reasons the answer might be "part of this should be rules, not a model", and name which
of the seven questions each reason comes from.

### Transfer to a different situation

**E12 (design).** For each of the following, decide: rule, query, experiment, optimisation, or
ML - and justify in one sentence.
(a) Which warehouse should ship this order to minimise total distance, given all stock levels?
(b) Will this customer open the email we are about to send?
(c) Did last quarter's price increase reduce order volume?
(d) Is this passport number in a valid format?
(e) How many support agents should be on shift next Tuesday at 14:00?

### Explain it to someone non-technical

**E13 (explain).** Tomás asks: *"why can't the AI just work out the loyalty policy from my
sales data?"* Answer him in under 70 words, without using the words model, algorithm, or data.

### Optional challenge

**E14 (design + code).** Build the hybrid from the remedies table: apply the *policy* to decide
free delivery, and train a model only on the part the policy does not determine - say,
predicting delivery duration in days from distance and weather. Show that the policy part stays
exact when the threshold changes while the learned part is untouched. What did the hybrid buy
you, and what did it cost?

In [ ]:
# Your workspace. Still in memory: orders, train, test, tree, logreg, policy,
# parcels, p_train, p_test, model, best_threshold, accuracy_score.

## Mastery check

Without scrolling up, can you:

- [ ] Draw the two-line diagram of what ordinary programming and ML each take in and give out?
      *(If not: "What machine learning actually is".)*
- [ ] Say why a perfect score in task A was an argument *against* the model? *(If not: task A.)*
- [ ] Recite at least five of the seven questions? *(If not: "The seven questions".)*
- [ ] Name the five alternatives to ML and one situation each wins in? *(If not: "What to use
      instead".)*
- [ ] Explain why a stale rule and a stale model fail identically but cost differently to fix?
      *(If not: "Failure lab".)*

## What should now feel instinctive

1. **"Can I just write this down?"** - asked before any thought about models.
2. **"Is this about the past or the future?"** Past means a query. A model that predicts
   something already recorded is a slow, approximate database.
3. **"Am I being asked to predict, or to find a cause?"** Different questions, different tools,
   and confusing them produces confident nonsense.
4. **"Do the answers exist in the data, as a column?"** Not the inputs - the *outcomes*.
5. **"What happens to this when the world changes?"** - asked before deploying, not after.

## Flashcards

| Question | Answer |
|---|---|
| What does ML invert about programming? | Programming: rules + data -> answers. ML: data + answers -> rules |
| The extra ingredient ML needs | Recorded outcomes - examples labelled with what actually happened |
| Why not model a known policy? | ML can at best reproduce it, at the cost of data, training and maintenance - and usually smooths its sharp edges |
| Define accuracy | The share of predictions that are correct |
| The seven questions, in short | Pattern? Writable rule? Labelled outcomes? Stable future? Errors tolerable? Repeated often? Inputs available at decision time? |
| Question about the past -> use? | A database query. The recorded answer beats an estimate of it |
| Question about a cause -> use? | An experiment, or causal methods - not a predictive model |
| You know the objective and constraints -> use? | Optimisation, not prediction |
| What is drift? | The world moving away from the data the model was fitted to, so a once-correct model quietly stops being correct |
| Stale rule vs stale model | Both wrong the same amount; the rule is one edit, the model needs new labelled data, retraining and redeployment |

## Next

**Chapter 00-03 · The kinds of learning, and the map of the work.**

You can now tell an ML problem from a query, a rule and an experiment. The next chapter splits
the ML side into its families - supervised, unsupervised, semi-supervised, self-supervised,
transfer and reinforcement learning - runs three of them on one small dataset, and lays out the
lifecycle so you can see how little of the work is the modelling. Then **00-04** takes on
Tomás's pastries by the door: the difference between predicting something and knowing what
would happen if you changed it.

New terms are in [GLOSSARY.md](../../GLOSSARY.md).